# Multi-Source Production Environment Test

This notebook exercises the production OpenRouter free LLM, OpenRouter embeddings, OpenSearch multi-index retrieval, MongoDB, and index aliases through the real application factories. Set `USER_ID` to a trusted test owner whose production data may be queried.

In [ ]:
import os
import sys
from pathlib import Path


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "app" / "api" / "main.py").is_file():
            return candidate
    raise RuntimeError("Repository root containing app/api/main.py was not found")


REPOSITORY_ROOT = find_repository_root(Path.cwd().resolve())
os.chdir(REPOSITORY_ROOT)
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))
print(f"Repository: {REPOSITORY_ROOT}")

## Load production configuration safely

Secrets are read from the process environment or the repository-root `.env`. Secret values are never printed; only safe endpoint metadata and OpenRouter key presence are displayed.

In [ ]:
from pprint import pprint
from urllib.parse import urlsplit

from app.config.settings import Settings


def safe_endpoint_description(endpoint: str) -> str:
    parsed = urlsplit(endpoint)
    scheme = parsed.scheme if parsed.scheme in {"http", "https"} else "configured"
    hostname = parsed.hostname or "configured-host"
    if ":" in hostname:
        hostname = f"[{hostname}]"
    try:
        port = parsed.port
    except ValueError:
        port = None
    if port is not None:
        hostname = f"{hostname}:{port}"
    return f"{scheme}://{hostname}"


settings = Settings.from_env()
llm_endpoint = settings.resolve_llm_endpoint()
embedding_endpoint = settings.resolve_embedding_endpoint()
safe_settings = {
    "llm_provider": llm_endpoint.provider,
    "llm_model": llm_endpoint.model,
    "llm_base_url": safe_endpoint_description(llm_endpoint.base_url),
    "embedding_provider": embedding_endpoint.provider,
    "embedding_model": embedding_endpoint.model,
    "embedding_base_url": safe_endpoint_description(embedding_endpoint.base_url),
    "mongo_db": settings.mongo_db,
    "mail_alias": settings.mail_index_alias,
    "calendar_alias": settings.calendar_index_alias,
    "timezone": settings.default_user_timezone,
    "MULTI_SOURCE_DEMO": settings.multi_source_demo,
    "OPENROUTER_API_KEY configured": bool(
        settings.openrouter_api_key.get_secret_value().strip()
    ),
}
pprint(safe_settings)
assert settings.multi_source_demo is False, "Set MULTI_SOURCE_DEMO=false"
assert llm_endpoint.provider == "openrouter"
assert llm_endpoint.model == "openrouter/free"
assert safe_settings["OPENROUTER_API_KEY configured"], (
    "Set OPENROUTER_API_KEY in the process environment or repository .env"
)

## Build and check the production application

Application construction and readiness use the same production factories and dependency checks as the API service. The in-process ASGI client avoids an external HTTP server while preserving the real `/ready` and `/v1/chat` paths.

In [ ]:
import httpx

from app.api.dependencies import build_container
from app.api.main import create_app


container = build_container(settings)
application = create_app(container)
transport = httpx.ASGITransport(app=application, raise_app_exceptions=False)
client = httpx.AsyncClient(transport=transport, base_url="http://production-notebook")

readiness_response = await client.get("/ready")
readiness_body = readiness_response.json()
print("readiness HTTP", readiness_response.status_code)
pprint(readiness_body)
assert readiness_response.status_code == 200, (
    "Production dependencies are not ready; fix the reported dependency before chat"
)
assert readiness_body.get("status") == "ready"

## Configure public request inputs

Edit the trusted owner, initial question, public team/week filters, and follow-up question below. No private infrastructure or routing fields belong in the request.

In [ ]:
USER_ID = "kim"
QUESTION = "이번 주 일정 뭐야?"
FILTERS = {
    "teams": [],
    "weeks": [],
}
FOLLOW_UP = "그중 첫 번째 일정의 상세 내용과 참석자를 알려줘"

In [ ]:
obsolete_fields = {"response_mode", "mode", "routing"}


def show_chat_result(response: httpx.Response) -> dict:
    body = response.json()
    print("HTTP", response.status_code)
    print("x-trace-id", response.headers.get("x-trace-id", "missing"))
    if response.status_code != 200:
        pprint(body)
        raise AssertionError("Chat request failed; inspect the safe error and trace ID")

    assert obsolete_fields.isdisjoint(body)
    print("answer:", body.get("answer"))
    print("references:")
    for reference in body.get("references", []):
        print(
            f"- [{reference['evidence_id']}] {reference['source_type']} | "
            f"{reference['title']} | {reference['excerpt']}"
        )
    print("disclosures:", body.get("disclosures", []))
    print("agent_trace:", body.get("agent_trace"))
    execution = body.get("execution") or {}
    print(
        "execution:",
        {
            "status": execution.get("status"),
            "search_count": execution.get("search_count"),
            "evidence_count": execution.get("evidence_count"),
            "duration_ms": execution.get("duration_ms"),
        },
    )
    return body


chat_request = {
    "user_id": USER_ID,
    "message": QUESTION,
    "filters": FILTERS,
}
assert obsolete_fields.isdisjoint(chat_request)
chat_response = await client.post("/v1/chat", json=chat_request)
chat_body = show_chat_result(chat_response)
conversation_id = chat_body["conversation_id"]
required_tools = {"search_calendar"}
assert required_tools <= set(chat_body["agent_trace"]["tool_calls"])
required_llm_calls = ["routing", "planner", "judge", "answer"]
assert chat_body["agent_trace"]["llm_calls"] == required_llm_calls
required_source_types = {"calendar"}
actual_source_types = {
    reference["source_type"] for reference in chat_body["references"]
}
assert required_source_types <= actual_source_types
assert chat_body["quality"]["citation_valid"] is True

## Validate conversation continuity

The follow-up reuses the server-issued conversation ID and the same owner, validating MongoDB-backed conversation continuity.

In [ ]:
follow_up_request = {
    "user_id": USER_ID,
    "message": FOLLOW_UP,
    "conversation_id": conversation_id,
    "filters": FILTERS,
}
assert obsolete_fields.isdisjoint(follow_up_request)
follow_up_response = await client.post("/v1/chat", json=follow_up_request)
follow_up_body = show_chat_result(follow_up_response)
assert follow_up_body["conversation_id"] == conversation_id

In [ ]:
await client.aclose()
print("Notebook HTTP client closed")